Open in colab: https://colab.research.google.com/github/Thanaritt-K/Thai-BPE-EXP/blob/main/BPEexperiment.ipynb

First, we need to install some dependencies used in this experiment

In [2]:
pip install --upgrade pip

Note: you may need to restart the kernel to use updated packages.


The data can be downloaded from https://github.com/korakot/corpus/tree/main/BEST or https://huggingface.co/datasets/nectec/best2009 But for simplicity I'll use the first one, since it's more easily managed. It has already been uploaded to this repo as well, for archival purposes.

In [50]:
import os
import numpy as np
import pandas as pd

# For quick experimentation, we will include only 200 texts
folderpath = r"BEST/TrainingSet/article/" 
files = os.listdir(folderpath)
files = np.sort(files)

files_txt = [i for i in files if i.endswith('.txt')]
training_data = []

for file_name in files_txt:
    text_file = open(folderpath+file_name,'r', encoding='utf8', errors='ignore')
    training_data.append(text_file.readlines())

# flatten the data structure
training_data = [
    x
    for xs in training_data
    for x in xs
]

An example of Thai sentences, with word boundaries annotation from BEST-2010 Corpus and number of sentences we will use to train the BPE segmentor

In [ ]:
print(training_data[:5])
len(training_data)

['กฎหมาย|กับ|การ|เบียดบัง|คน|จน|\n', 'จาก|ต้นฉบับ|เรื่อง| |"|บท|นำ|:| |คน|จน|ภาย|ใต้|ความ|สัมพันธ์|ทาง|กฎหมาย|"|\n', '<NE>ไพสิฐ พาณิชย์กุล</NE>|\n', 'อาจารย์|ประจำ| |<NE>สาขาวิชานิติศาสตร์</NE>| |<NE>ภาควิชารัฐศาสตร์</NE>| |<NE>คณะสังคมศาสตร์</NE>| |<NE>มหาวิทยาลัยเชียงใหม่</NE>|\n', '(|บทความ|นี้|ยาว|ประมาณ| |9| |หน้า|กระดาษ| |A|4|)|\n']


16990

Next, we substitues whitespaces with a meta symbol _ (U+2581), remove tags and some punctuation marks, such as quotation marks and parentheses, and then substitues word boundaries (|) with whitespace.

In [ ]:
import re

i = 0
training_data_processed = []
while i < len(training_data):
    text = training_data[i]
    text = re.sub(r'</?(AB|NE)>', '', text) #remove tags
    text = re.sub(r'[\"\:\(\)]', '', text) #remove some punctuation marks
    text = re.sub(r' ', '_', text) #substitute whitespaces to preserve them
    text = re.sub(r'\|', ' ', text) #substitute word boundaries to prepare the data for BPE training
    text = re.sub(r' {2,}', ' ', text) #remove double whitespace, but keeping newlines
    training_data_processed.append(text)
    i = i + 1
    

An example of Thai sentences after pre-processing

In [ ]:
print(training_data_processed[:5])

['กฎหมาย กับ การ เบียดบัง คน จน \n', 'จาก ต้นฉบับ เรื่อง _ บท นำ _ คน จน ภาย ใต้ ความ สัมพันธ์ ทาง กฎหมาย \n', 'ไพสิฐ_พาณิชย์กุล \n', 'อาจารย์ ประจำ _ สาขาวิชานิติศาสตร์ _ ภาควิชารัฐศาสตร์ _ คณะสังคมศาสตร์ _ มหาวิทยาลัยเชียงใหม่ \n', ' บทความ นี้ ยาว ประมาณ _ 9 _ หน้า กระดาษ _ A 4 \n']


16990